# Fixation mRNN — 1D inter-region bottleneck ensemble

Multi-initialization fit of the fixation mRNN with:

- **Within-region L1 regularization** (`l1_weight_scale = 0.01`) on the dense within-region recurrent blocks.
- **A rank-1 across-region bottleneck** (`recurrent_bottleneck_dim = 1`): every inter-region recurrent block is factorized as `left @ right` with a single shared latent, so all cross-region communication is squeezed through one dimension.

**What this notebook does**

1. Builds a seed plan of `N_INITIALIZATIONS` seeds (parametric).
2. Submits a dSQ **array job** (partition `psych_gpu`, 8G RAM, 4h wall-clock per task) for any initialization whose checkpoint is missing, and stores the SLURM job id.
3. On re-run it checks completeness: if some outputs are missing it looks up the stored job id and **prints that the job is still running** rather than resubmitting; only when no job is running does it (re)submit the missing tasks.
4. Once **all** checkpoints exist it moves on to analysis: best-fit inspection, all-fit comparison, and cross-initialization consistency of within/inter-regional currents and geometry.

Run this with the `gaze_processing` conda environment.


## 0 · Environment and imports

In [1]:
from __future__ import annotations

from dataclasses import replace, asdict
from datetime import datetime, timezone
from io import BytesIO
from pathlib import Path
import json
import shlex
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from IPython.display import Image, display
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize

repo_root = Path.cwd()
if not (repo_root / "src").exists():
    repo_root = next(parent for parent in Path.cwd().parents if (parent / "src").exists())
if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

from dal_monte_2022_analysis.ephys.modeling import (
    backproject_replay_outputs_to_firing_rates,
    extract_fixation_latent_dynamics,
    extract_region_current_vectors,
    load_fixation_mrnn_config,
    make_targets,
    pc_reconstructed_firing_rate_accuracy,
    reconstruction_accuracy,
    replay_fixation_mrnn_run,
    resolve_fixation_mrnn_output_root,
    settings_from_config,
)
from dal_monte_2022_analysis.ephys.modeling.fixation_mrnn_training import load_or_create_seed_plan
from dal_monte_2022_analysis.runtime.hpc.jobs import submit_dsq_array_job, write_job_file

plt.rcParams.update({"figure.dpi": 140, "axes.spines.top": False, "axes.spines.right": False})


def display_figure(fig, *, dpi=150):
    buffer = BytesIO()
    fig.savefig(buffer, format="png", dpi=dpi, bbox_inches="tight")
    plt.close(fig)
    buffer.seek(0)
    display(Image(data=buffer.getvalue()))


print("repo_root:", repo_root)


repo_root: /gpfs/milgram/pi/chang/pg496/repositories/dal_monte_2022_analysis


## 1 · Experiment settings and generated config

The exact network settings requested for this run are set below. The training CLI
(`scripts/ephys/modeling/train_fixation_mrnn.py`) does **not** expose flags for the
bottleneck dimension, connectivity mode, spectral radius, activation, loss function, or
`pca_n_components`, so we bake those into a dedicated config YAML written to the job
directory. Every job task reads that config, guaranteeing all initializations share the
exact architecture below.

In [2]:
# ---- parametric knobs -------------------------------------------------------
N_INITIALIZATIONS = 10          # number of random initializations (seeds)
RECURRENT_BOTTLENECK_DIM = 1    # rank of the across-region bottleneck

SCRATCH_ID = "bottleneck1d_l1within_50u_100k_10init"

cfg = load_fixation_mrnn_config(repo_root / "configs" / "ephys_fixation_mrnn.yaml")

# The requested network settings (see task spec).
experiment_overrides = {
    "device": "auto",
    "target_mode": "region_pcs",
    "pca_n_components": 42,
    "temporal_basis_count": 0,
    "hidden_units": 50,
    "activation": "tanh",
    "rec_constrained": False,
    "inp_constrained": False,
    "spectral_radius": 1.1,
    "recurrent_connectivity": "full",          # within-region blocks + inter-region bottleneck
    "recurrent_bottleneck_dim": RECURRENT_BOTTLENECK_DIM,
    "epochs": 100_000,
    "lr": 3e-4,
    "loss_fn": "mse",
    "temporal_derivative_loss_scale": 1.0,
    "temporal_curvature_loss_scale": 0.5,
    "correlation_loss_scale": 0.0,
    "variance_loss_scale": 0.0,
    "fr_reconstruction_loss_scale": 0.0,
    "fr_temporal_derivative_loss_scale": 0.0,
    "fr_temporal_curvature_loss_scale": 0.0,
    "l1_weight_scale": 0.01,                    # within-region recurrent L1
    "l1_rate_scale": 0.0,
    "l2_weight_scale": 0.0,
    "l2_rate_scale": 0.0,
    "gradient_clip_norm": 1.0,
    # divergence guardrails so a bad init fails fast and the CLI retries a fresh seed
    "divergence_loss_threshold": 1e3,
    "divergence_patience": 100,
    "divergence_min_iteration": 100,
    "initialization_mode": "single",
}

# In-notebook settings object (device forced to auto so analysis replay works anywhere).
settings = settings_from_config(cfg, overrides={**experiment_overrides, "device": "auto"})
settings = replace(settings, dataset_cfg_path=str(repo_root / "configs" / "dataset.yaml"))
settings


FixationMRNNRunSettings(dataset_cfg_path='/gpfs/milgram/pi/chang/pg496/repositories/dal_monte_2022_analysis/configs/dataset.yaml', input_subdir='ephys/psth/fixation_psth_averages', dataframe_filename='fixations_psth_10ms_combined_window_neg500ms_to_pos500ms.pkl', timeline_filename='fixations_psth_10ms_bin_centers_s_rel_window_neg500ms_to_pos500ms.pkl', output_subdir='ephys/modeling/fixation_mrnn', region_order=('ofc', 'bla', 'dmpfc', 'accg'), condition_order=('face_interactive', 'face_non_interactive', 'object'), target_mode='region_pcs', normalize_targets=True, normalization_stabilizer=5.0, pca_variance_threshold=0.95, pca_n_components=42, temporal_basis_count=0, hidden_units=50, activation='tanh', spectral_radius=1.1, rec_constrained=False, inp_constrained=False, recurrent_connectivity='full', recurrent_bottleneck_dim=1, batch_first=True, inp_noise=0.0, act_noise=0.0, epochs=100000, lr=0.0003, loss_fn='mse', temporal_derivative_loss_scale=1.0, temporal_curvature_loss_scale=0.5, corre

In [3]:
# ---- generate the job config YAML -------------------------------------------
job_dir = repo_root / "hpc" / "fixation_mrnn_1d_bottleneck"
log_dir = job_dir / "logs"
job_dir.mkdir(parents=True, exist_ok=True)

generated_cfg = dict(cfg)
generated_cfg.update({k: v for k, v in experiment_overrides.items()})
generated_cfg["device"] = "cuda"          # jobs run on the GPU partition
generated_cfg["scratch_id"] = SCRATCH_ID
generated_cfg["dataset_cfg_path"] = "configs/dataset.yaml"

config_path = job_dir / "config_1d_bottleneck.yaml"
with config_path.open("w", encoding="utf-8") as f:
    yaml.safe_dump(generated_cfg, f, sort_keys=False)

print("wrote", config_path)
print(config_path.read_text())


wrote /gpfs/milgram/pi/chang/pg496/repositories/dal_monte_2022_analysis/hpc/fixation_mrnn_1d_bottleneck/config_1d_bottleneck.yaml
dataset_cfg_path: configs/dataset.yaml
input_subdir: ephys/psth/fixation_psth_averages
dataframe_filename: fixations_psth_10ms_combined_window_neg500ms_to_pos500ms.pkl
timeline_filename: fixations_psth_10ms_bin_centers_s_rel_window_neg500ms_to_pos500ms.pkl
output_subdir: ephys/modeling/fixation_mrnn
scratch_id: bottleneck1d_l1within_50u_100k_10init
region_order:
- ofc
- bla
- dmpfc
- accg
condition_order:
- face_interactive
- face_non_interactive
- object
pca_variance_threshold: 0.95
normalize_targets: true
normalization_stabilizer: 5.0
temporal_basis_count: 0
hidden_units: 50
activation: tanh
spectral_radius: 1.1
recurrent_bottleneck_dim: 1
batch_first: true
rec_constrained: false
inp_constrained: false
inp_noise: 0.0
act_noise: 0.0
target_mode: region_pcs
epochs: 100000
lr: 0.0003
loss_fn: mse
temporal_derivative_loss_scale: 1.0
temporal_curvature_loss_sca

## 2 · Seed plan and run table

The seed plan is persisted once (`seed_plan.json`) inside the ensemble directory so the
set of initializations is stable across notebook restarts and job resubmissions. Each
initialization gets its own run directory `init=NNN_seed=SSSS/`.

In [4]:
output_root = resolve_fixation_mrnn_output_root(settings) / "scratch"
ensemble_dir = output_root / SCRATCH_ID
ensemble_dir.mkdir(parents=True, exist_ok=True)

seed_plan = load_or_create_seed_plan(
    ensemble_dir,
    settings,
    n_seeds=N_INITIALIZATIONS,
    overwrite=False,
)


def build_run_table():
    rows = []
    for idx, seed in enumerate(seed_plan):
        run_dir = ensemble_dir / f"init={idx:03d}_seed={int(seed)}"
        rows.append(
            {
                "init_idx": idx,
                "seed": int(seed),
                "run_dir": str(run_dir),
                "checkpoint_path": str(run_dir / "checkpoint_final.pth"),
                "history_path": str(run_dir / "history.csv"),
            }
        )
    table = pd.DataFrame(rows)
    table["is_complete"] = table["checkpoint_path"].map(lambda p: Path(p).exists())
    table["history_exists"] = table["history_path"].map(lambda p: Path(p).exists())
    return table


run_table = build_run_table()
n_complete = int(run_table["is_complete"].sum())
print(f"Ensemble dir: {ensemble_dir}")
print(f"Complete checkpoints: {n_complete} / {N_INITIALIZATIONS}")
run_table


Ensemble dir: /gpfs/milgram/pi/chang/pg496/repositories/local_data/dal_monte_2022/analysis_outputs/ephys/modeling/fixation_mrnn/scratch/bottleneck1d_l1within_50u_100k_10init
Complete checkpoints: 0 / 10


,init_idx,seed,run_dir,checkpoint_path,history_path,is_complete,history_exists
0,0,211382628,/gpfs/milgram/pi/chang/pg496/repositories/loca...,/gpfs/milgram/pi/chang/pg496/repositories/loca...,/gpfs/milgram/pi/chang/pg496/repositories/loca...,False,False
1,1,1366902869,/gpfs/milgram/pi/chang/pg496/repositories/loca...,/gpfs/milgram/pi/chang/pg496/repositories/loca...,/gpfs/milgram/pi/chang/pg496/repositories/loca...,False,False
2,2,1751793295,/gpfs/milgram/pi/chang/pg496/repositories/loca...,/gpfs/milgram/pi/chang/pg496/repositories/loca...,/gpfs/milgram/pi/chang/pg496/repositories/loca...,False,False
3,3,826376757,/gpfs/milgram/pi/chang/pg496/repositories/loca...,/gpfs/milgram/pi/chang/pg496/repositories/loca...,/gpfs/milgram/pi/chang/pg496/repositories/loca...,False,False
4,4,802808611,/gpfs/milgram/pi/chang/pg496/repositories/loca...,/gpfs/milgram/pi/chang/pg496/repositories/loca...,/gpfs/milgram/pi/chang/pg496/repositories/loca...,False,False
5,5,101888267,/gpfs/milgram/pi/chang/pg496/repositories/loca...,/gpfs/milgram/pi/chang/pg496/repositories/loca...,/gpfs/milgram/pi/chang/pg496/repositories/loca...,False,False
6,6,137839924,/gpfs/milgram/pi/chang/pg496/repositories/loca...,/gpfs/milgram/pi/chang/pg496/repositories/loca...,/gpfs/milgram/pi/chang/pg496/repositories/loca...,False,False
7,7,2051389637,/gpfs/milgram/pi/chang/pg496/repositories/loca...,/gpfs/milgram/pi/chang/pg496/repositories/loca...,/gpfs/milgram/pi/chang/pg496/repositories/loca...,False,False
8,8,473707325,/gpfs/milgram/pi/chang/pg496/repositories/loca...,/gpfs/milgram/pi/chang/pg496/repositories/loca...,/gpfs/milgram/pi/chang/pg496/repositories/loca...,False,False
9,9,1945729569,/gpfs/milgram/pi/chang/pg496/repositories/loca...,/gpfs/milgram/pi/chang/pg496/repositories/loca...,/gpfs/milgram/pi/chang/pg496/repositories/loca...,False,False


## 3 · Job submission / status logic

Decision tree (re-runnable and idempotent):

- **All checkpoints present** → skip submission, drop through to analysis.
- **Some/all missing** →
  - If a previously stored job id is **still running** (any array task `PENDING`/`RUNNING`/`CONFIGURING`), print that and do **not** resubmit.
  - Otherwise submit a fresh dSQ array job for exactly the missing initializations, and persist the new job id.

Set `SUBMIT_MISSING_JOBS = True` to actually submit. Resources per task: `psych_gpu`,
`gpu:1`, 1 CPU × 8G RAM, 4h wall-clock.

In [5]:
# ---- resource + submission configuration ------------------------------------
SUBMIT_MISSING_JOBS = True          # set False to preview commands without submitting

PARTITION = "psych_gpu"
GRES = "gpu:1"
CPUS_PER_TASK = 1
MEM_PER_CPU = "8G"                  # 8 GB RAM per task
TIME_LIMIT = "04:00:00"            # 4 hours per task
MAX_DIVERGENCE_RETRIES = 5

job_file_path = job_dir / "joblist.txt"
sbatch_script_path = job_dir / "submit.sbatch"
job_record_path = job_dir / "last_submission.json"


def build_training_command(row):
    scratch_child = f"{SCRATCH_ID}/init={int(row.init_idx):03d}_seed={int(row.seed)}"
    argv = [
        "python", "-u", "scripts/ephys/modeling/train_fixation_mrnn.py",
        "--mrnn-cfg", str(config_path),
        "--scratch-id", scratch_child,
        "--seed", str(int(row.seed)),
        "--initialization-mode", "single",
        "--device", "cuda",
        # redundant explicit overrides for the CLI-exposed training params
        "--target-mode", "region_pcs",
        "--epochs", str(settings.epochs),
        "--lr", str(settings.lr),
        "--hidden-units", str(settings.hidden_units),
        "--temporal-basis-count", str(settings.temporal_basis_count),
        "--temporal-derivative-loss-scale", str(settings.temporal_derivative_loss_scale),
        "--temporal-curvature-loss-scale", str(settings.temporal_curvature_loss_scale),
        "--correlation-loss-scale", "0",
        "--variance-loss-scale", "0",
        "--fr-reconstruction-loss-scale", "0",
        "--fr-temporal-derivative-loss-scale", "0",
        "--fr-temporal-curvature-loss-scale", "0",
        "--l1-weight-scale", str(settings.l1_weight_scale),
        "--l1-rate-scale", "0",
        "--l2-weight-scale", "0",
        "--l2-rate-scale", "0",
        "--gradient-clip-norm", str(settings.gradient_clip_norm),
        "--divergence-loss-threshold", str(settings.divergence_loss_threshold),
        "--divergence-patience", str(settings.divergence_patience),
        "--divergence-min-iteration", str(settings.divergence_min_iteration),
        "--max-divergence-retries", str(MAX_DIVERGENCE_RETRIES),
    ]
    segments = [
        "module load miniconda",
        "module load CUDA/12.1.1",
        "conda deactivate || true",
        "conda activate gaze_processing",
        f"cd {shlex.quote(str(repo_root))}",
        shlex.join(argv),
    ]
    return " && ".join(segments)


def job_array_active_tasks(job_id):
    """Return (n_active, statuses) for a SLURM job id, or (None, None) if unknown."""
    if not job_id:
        return None, None
    result = subprocess.run(
        ["squeue", "--job", str(job_id), "-h", "-o", "%T"],
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        return None, None
    statuses = result.stdout.split()
    active = [s for s in statuses if s in {"PENDING", "RUNNING", "CONFIGURING"}]
    return len(active), statuses


def load_job_record():
    if job_record_path.exists():
        with job_record_path.open("r", encoding="utf-8") as f:
            return json.load(f)
    return None


def save_job_record(job_id, missing_rows):
    payload = {
        "job_id": str(job_id),
        "submitted_at_utc": datetime.now(timezone.utc).isoformat(),
        "scratch_id": SCRATCH_ID,
        "n_missing": int(len(missing_rows)),
        "missing_init_idxs": [int(i) for i in missing_rows["init_idx"].tolist()],
        "partition": PARTITION,
    }
    with job_record_path.open("w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2, sort_keys=True)
    return payload


In [6]:
# ---- run the decision tree --------------------------------------------------
run_table = build_run_table()
missing = run_table.loc[~run_table["is_complete"]].copy()
all_complete = missing.empty

stored_record = load_job_record()
stored_job_id = stored_record["job_id"] if stored_record else None
active_count, active_statuses = job_array_active_tasks(stored_job_id)

active_job_id = None

if all_complete:
    print(f"All {N_INITIALIZATIONS} checkpoints exist -> proceeding to analysis.")
    if stored_job_id and active_count:
        print(f"(Stored job {stored_job_id} still shows {active_count} active task(s); safe to ignore.)")
elif stored_job_id and active_count:
    # A tracked job is still running: do not resubmit.
    active_job_id = stored_job_id
    print(
        f"{len(missing)} / {N_INITIALIZATIONS} initializations still missing, "
        f"but stored job {stored_job_id} is STILL RUNNING "
        f"({active_count} active task(s): {sorted(set(active_statuses))}).\n"
        f"Not resubmitting. Re-run this notebook once the job finishes."
    )
    display(missing[["init_idx", "seed", "run_dir"]].head(20))
else:
    # Nothing running (or no record / job finished) and outputs are missing -> submit.
    if stored_job_id:
        print(f"Stored job {stored_job_id} is no longer running; {len(missing)} initializations still missing.")
    else:
        print(f"No prior job on record; {len(missing)} initializations missing.")
    commands = [build_training_command(row) for row in missing.itertuples(index=False)]
    write_job_file(job_file_path, commands)
    print(f"Wrote {len(commands)} task(s) to {job_file_path}")
    display(missing[["init_idx", "seed", "run_dir"]].head(20))
    if SUBMIT_MISSING_JOBS:
        active_job_id = submit_dsq_array_job(
            job_file_path=job_file_path,
            sbatch_script_path=sbatch_script_path,
            log_dir=log_dir,
            job_name="fix_mrnn_1d_bottleneck_missing",
            partition=PARTITION,
            cpus_per_task=CPUS_PER_TASK,
            mem_per_cpu=MEM_PER_CPU,
            time_limit=TIME_LIMIT,
            gres=GRES,
        )
        record = save_job_record(active_job_id, missing)
        print("Stored job record:", record)
    else:
        print("SUBMIT_MISSING_JOBS is False -> not submitted. Inspect the joblist above.")

print("\nactive_job_id:", active_job_id, "| all_complete:", all_complete)


No prior job on record; 10 initializations missing.
Wrote job file to /gpfs/milgram/pi/chang/pg496/repositories/dal_monte_2022_analysis/hpc/fixation_mrnn_1d_bottleneck/joblist.txt
Wrote 10 task(s) to /gpfs/milgram/pi/chang/pg496/repositories/dal_monte_2022_analysis/hpc/fixation_mrnn_1d_bottleneck/joblist.txt


,init_idx,seed,run_dir
0,0,211382628,/gpfs/milgram/pi/chang/pg496/repositories/loca...
1,1,1366902869,/gpfs/milgram/pi/chang/pg496/repositories/loca...
2,2,1751793295,/gpfs/milgram/pi/chang/pg496/repositories/loca...
3,3,826376757,/gpfs/milgram/pi/chang/pg496/repositories/loca...
4,4,802808611,/gpfs/milgram/pi/chang/pg496/repositories/loca...
5,5,101888267,/gpfs/milgram/pi/chang/pg496/repositories/loca...
6,6,137839924,/gpfs/milgram/pi/chang/pg496/repositories/loca...
7,7,2051389637,/gpfs/milgram/pi/chang/pg496/repositories/loca...
8,8,473707325,/gpfs/milgram/pi/chang/pg496/repositories/loca...
9,9,1945729569,/gpfs/milgram/pi/chang/pg496/repositories/loca...


Batch script generated. To submit your jobs, run:
 sbatch /gpfs/milgram/pi/chang/pg496/repositories/dal_monte_2022_analysis/hpc/fixation_mrnn_1d_bottleneck/submit.sbatch
Submitted job array with ID: 28992754
Stored job record: {'job_id': '28992754', 'submitted_at_utc': '2026-07-10T20:06:52.577577+00:00', 'scratch_id': 'bottleneck1d_l1within_50u_100k_10init', 'n_missing': 10, 'missing_init_idxs': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9], 'partition': 'psych_gpu'}

active_job_id: 28992754 | all_complete: False


> **Checkpoint.** If jobs were just submitted (or are still running), stop here and
> re-run the notebook from the top once they finish. The analysis section below only
> executes when every initialization has a checkpoint. You can poll the job with
> `squeue --job <id>` in a terminal, or just re-run this notebook.

## 4 · Load completed runs and rank fits

In [7]:
run_table = build_run_table()
complete = run_table[run_table["is_complete"] & run_table["history_exists"]].copy()
all_complete = bool((~run_table["is_complete"]).sum() == 0)
ANALYSIS_READY = all_complete and len(complete) >= 2

print(f"Completed runs available: {len(complete)} / {N_INITIALIZATIONS}")
if not ANALYSIS_READY:
    print(
        "Analysis is gated until ALL initializations are complete.\n"
        "Re-run once the array job finishes (need >= 2 runs for consistency metrics)."
    )


Completed runs available: 0 / 10
Analysis is gated until ALL initializations are complete.
Re-run once the array job finishes (need >= 2 runs for consistency metrics).


In [8]:
# ---- fit-quality ranking helpers (transient-aware, from the ensemble workflow) --
LOSS_TRANSIENT_JUMP_LOG10 = 0.5


def read_run_seed(run_dir, fallback_seed):
    manifest_path = Path(run_dir) / "manifest.json"
    if manifest_path.exists():
        with manifest_path.open("r", encoding="utf-8") as f:
            manifest = json.load(f)
        if manifest.get("status", "complete") != "failed" and "seed" in manifest:
            return int(manifest["seed"])
    return int(fallback_seed)


def detect_loss_transients(history, *, jump_threshold_log10=LOSS_TRANSIENT_JUMP_LOG10):
    hist = history.sort_values("iteration")
    losses = hist["loss"].to_numpy(dtype=float)
    iterations = hist["iteration"].to_numpy(dtype=float)
    finite = np.isfinite(losses) & (losses > 0)
    safe = np.where(finite, losses, np.nan)
    log_loss = np.log10(np.clip(safe, 1e-12, None))
    delta = np.diff(log_loss)
    excess = np.where(np.isfinite(delta), np.maximum(delta - float(jump_threshold_log10), 0.0), 0.0)
    transient_idx = np.flatnonzero(excess > 0) + 1
    return {"losses": losses, "iterations": iterations, "safe": safe,
            "delta": delta, "excess": excess, "transient_idx": transient_idx}


def loss_fit_metrics(history, *, total_iterations):
    d = detect_loss_transients(history)
    safe = d["safe"]
    iterations = d["iterations"]
    if safe.size == 0 or not np.isfinite(safe).any():
        return {"final_loss": np.inf, "final_iteration": 0, "completion_fraction": 0.0,
                "n_loss_transients": 0, "transient_penalty": np.inf, "fit_quality_score": np.inf}
    transient_idx = d["transient_idx"]
    progress = np.clip(iterations[1:] / float(total_iterations), 0.0, 1.0)
    transient_penalty = float(np.sum(d["excess"] * progress**2))
    final_loss = float(safe[-1])
    log_final = float(np.log10(max(final_loss, 1e-12)))
    incomplete_penalty = 5.0 * max(0.0, 1.0 - float(iterations[-1]) / float(total_iterations))
    return {
        "final_loss": final_loss,
        "final_iteration": int(iterations[-1]),
        "completion_fraction": float(iterations[-1] / float(total_iterations)),
        "n_loss_transients": int(transient_idx.size),
        "transient_penalty": transient_penalty,
        "fit_quality_score": log_final + transient_penalty + incomplete_penalty,
    }


if ANALYSIS_READY:
    history_by_init = {}
    rank_rows = []
    for row in complete.itertuples(index=False):
        hist = pd.read_csv(row.history_path)
        history_by_init[int(row.init_idx)] = hist
        rank_rows.append({
            "init_idx": int(row.init_idx),
            "seed": read_run_seed(row.run_dir, row.seed),
            "planned_seed": int(row.seed),
            "run_dir": str(row.run_dir),
            **loss_fit_metrics(hist, total_iterations=settings.epochs),
        })
    model_rank = pd.DataFrame(rank_rows).sort_values("fit_quality_score").reset_index(drop=True)
    model_rank["rank"] = np.arange(1, len(model_rank) + 1)
    best_init_idx = int(model_rank.iloc[0]["init_idx"])
    best_run_dir = Path(model_rank.iloc[0]["run_dir"])
    display(model_rank[["rank", "init_idx", "seed", "final_loss", "completion_fraction",
                        "n_loss_transients", "transient_penalty", "fit_quality_score"]].round(4))
    print("Best initialization:", best_init_idx, "->", best_run_dir.name)


In [9]:
if ANALYSIS_READY:
    targets = make_targets(settings)
    timeline = np.asarray(targets.timeline_s, dtype=float)
    target_fr_by_region = targets.pc_reconstructed_raw_by_region()

    replay_cache = {}

    def load_replay(init_idx):
        init_idx = int(init_idx)
        if init_idx not in replay_cache:
            run_dir = Path(model_rank.loc[model_rank["init_idx"] == init_idx, "run_dir"].iloc[0])
            replay_cache[init_idx] = replay_fixation_mrnn_run(run_dir, device="cpu")
        return replay_cache[init_idx]

    reference_replay = load_replay(best_init_idx)
    region_order = tuple(reference_replay["region_order"])
    condition_order = tuple(reference_replay["condition_order"])
    region_colors = {"ofc": "#4c78a8", "bla": "#f58518", "dmpfc": "#54a24b", "accg": "#e45756"}
    condition_colors = {"face_interactive": "#b64198", "face_non_interactive": "#4c9a2a", "object": "#6f4e37"}
    print("regions:", region_order, "| conditions:", condition_order)


## 5 · Best-fit network

Loss trajectory, within/inter-regional weight distributions, region-PC reconstruction,
and PC-backprojected firing-rate reconstruction for the top-ranked initialization.

In [10]:
def plot_loss_trajectory(hist, *, title):
    fig, ax = plt.subplots(figsize=(8.0, 3.8))
    weighted = {
        "reconstruction_loss": 1.0,
        "temporal_derivative_loss": settings.temporal_derivative_loss_scale,
        "temporal_curvature_loss": settings.temporal_curvature_loss_scale,
        "weight_loss": 1.0,  # already scaled inside training
    }
    for column, weight in weighted.items():
        if column not in hist.columns or float(weight) == 0.0:
            continue
        label = column.replace("_loss", "")
        ax.plot(hist["iteration"], hist[column] * float(weight), linewidth=1.0, label=label)
    ax.plot(hist["iteration"], hist["loss"], color="black", linewidth=1.6, label="total")
    ax.set(title=title, xlabel="iteration", ylabel="loss (log)")
    ax.set_yscale("log")
    ax.legend(frameon=False, fontsize=8, ncol=2)
    fig.tight_layout()
    return fig


if ANALYSIS_READY:
    display_figure(plot_loss_trajectory(
        history_by_init[best_init_idx],
        title=f"Best fit (init {best_init_idx:03d}) — loss trajectory",
    ))


In [11]:
def plot_weight_distributions(model, *, title):
    within_vals = np.concatenate([
        model._within_region_block(r).detach().cpu().numpy().ravel() for r in model.region_order
    ])
    factor_vals, block_vals = [], []
    for (src, tgt), left in model._inter_region_left_params.items():
        right = model._inter_region_right_params[(src, tgt)]
        factor_vals.append(left.detach().cpu().numpy().ravel())
        factor_vals.append(right.detach().cpu().numpy().ravel())
        block_vals.append((left @ right).detach().cpu().numpy().ravel())
    factor_vals = np.concatenate(factor_vals) if factor_vals else np.array([])
    block_vals = np.concatenate(block_vals) if block_vals else np.array([])

    fig, axes = plt.subplots(1, 3, figsize=(13.5, 3.6))
    axes[0].hist(within_vals, bins=45, color="#4c78a8", alpha=0.85)
    axes[0].set(title="Within-region recurrent weights\n(L1-regularized)", xlabel="weight", ylabel="count")
    axes[0].axvline(0.0, color="black", lw=0.6, alpha=0.5)
    axes[1].hist(factor_vals, bins=45, color="#f58518", alpha=0.85)
    axes[1].set(title="Inter-region bottleneck factors\n(left / right, rank-1)", xlabel="factor value", ylabel="count")
    axes[1].axvline(0.0, color="black", lw=0.6, alpha=0.5)
    axes[2].hist(block_vals, bins=45, color="#e45756", alpha=0.85)
    axes[2].set(title="Effective inter-region block\n(left @ right)", xlabel="weight", ylabel="count")
    axes[2].axvline(0.0, color="black", lw=0.6, alpha=0.5)
    fig.suptitle(title, y=1.03)
    fig.tight_layout()
    return fig


if ANALYSIS_READY:
    display_figure(plot_weight_distributions(
        reference_replay["model"],
        title=f"Best fit (init {best_init_idx:03d}) — recurrent weight distributions",
    ))


In [12]:
def plot_pc_reconstruction(replay, *, region, title, max_pcs=4):
    n_pcs = min(max_pcs, targets.pcs_by_region[region].shape[-1])
    fig, axes = plt.subplots(n_pcs, len(condition_order),
                             figsize=(3.1 * len(condition_order), 1.7 * n_pcs),
                             sharex=True, squeeze=False)
    yhat = replay["output_by_region"][region].detach().cpu().numpy()
    for pc_idx in range(n_pcs):
        for col, condition in enumerate(condition_order):
            ax = axes[pc_idx, col]
            cond_idx = condition_order.index(condition)
            ax.plot(timeline, targets.pcs_by_region[region][cond_idx, :, pc_idx], color="black", lw=1.8, label="target")
            ax.plot(timeline, yhat[cond_idx, :, pc_idx], color="#2f6fbb", lw=1.1, label="model")
            ax.axvline(0.0, color="0.5", lw=0.6)
            if pc_idx == 0:
                ax.set_title(condition, fontsize=8)
            if col == 0:
                ax.set_ylabel(f"PC{pc_idx + 1}")
            if pc_idx == n_pcs - 1:
                ax.set_xlabel("time (s)")
    axes[0, -1].legend(frameon=False, fontsize=7)
    fig.suptitle(title, y=1.01)
    fig.tight_layout()
    return fig


def plot_fr_reconstruction(replay, *, region, unit_indices, title):
    predicted_fr = backproject_replay_outputs_to_firing_rates(replay)[region]
    fig, axes = plt.subplots(len(unit_indices), len(condition_order),
                             figsize=(3.1 * len(condition_order), 1.55 * len(unit_indices)),
                             sharex=True, squeeze=False)
    for row, unit_idx in enumerate(unit_indices):
        for col, condition in enumerate(condition_order):
            ax = axes[row, col]
            cond_idx = condition_order.index(condition)
            ax.plot(timeline, target_fr_by_region[region][cond_idx, :, unit_idx], color="black", lw=1.8, label="target")
            ax.plot(timeline, predicted_fr[cond_idx, :, unit_idx], color="#2f6fbb", lw=1.1, label="model")
            ax.axvline(0.0, color="0.5", lw=0.6)
            if row == 0:
                ax.set_title(condition, fontsize=8)
            if col == 0:
                ax.set_ylabel(f"unit {unit_idx}")
            if row == len(unit_indices) - 1:
                ax.set_xlabel("time (s)")
    axes[0, -1].legend(frameon=False, fontsize=7)
    fig.suptitle(title, y=1.01)
    fig.tight_layout()
    return fig


if ANALYSIS_READY:
    metrics = pd.concat([
        reconstruction_accuracy(reference_replay).assign(metric_space="region_pcs"),
        pc_reconstructed_firing_rate_accuracy(reference_replay).assign(metric_space="backprojected_fr"),
    ], ignore_index=True)
    fit_quality = (metrics.groupby(["metric_space", "region"], as_index=False)
                   .agg(mean_mse=("mse", "mean"), mean_r2=("r2", "mean"), mean_corr=("correlation", "mean")))
    display(fit_quality.round(4))

    diag_rng = np.random.default_rng(best_init_idx)
    for region in region_order:
        display_figure(plot_pc_reconstruction(
            reference_replay, region=region,
            title=f"Best fit — {region.upper()} region-PC reconstruction"))
    fr_region = str(diag_rng.choice(region_order))
    n_units = target_fr_by_region[fr_region].shape[-1]
    unit_indices = np.sort(diag_rng.choice(n_units, size=min(6, n_units), replace=False))
    display_figure(plot_fr_reconstruction(
        reference_replay, region=fr_region, unit_indices=unit_indices,
        title=f"Best fit — {fr_region.upper()} backprojected firing-rate reconstruction"))


## 6 · Compare all initializations

Total-loss trajectories for every initialization, colour-coded from **best (dark) to
worst (light)** by final fit-quality rank, plus the sorted final-loss spectrum.

In [13]:
if ANALYSIS_READY:
    order = model_rank.sort_values("rank")
    cmap = plt.get_cmap("viridis")
    norm = Normalize(vmin=1, vmax=len(order))

    fig, axes = plt.subplots(1, 2, figsize=(13.0, 4.2))
    for _, row in order.iterrows():
        hist = history_by_init[int(row.init_idx)].sort_values("iteration")
        color = cmap(norm(int(row["rank"])))
        alpha = 0.95 if int(row["rank"]) <= 3 else 0.55
        axes[0].plot(hist["iteration"], hist["loss"], color=color, lw=1.2, alpha=alpha)
    axes[0].set(title="Total loss — all initializations (best=dark)", xlabel="iteration", ylabel="loss (log)")
    axes[0].set_yscale("log")
    sm = ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=axes[0])
    cbar.set_label("fit rank (1 = best)")

    ranks = order["rank"].to_numpy()
    finals = order["final_loss"].to_numpy()
    axes[1].scatter(ranks, finals, c=[cmap(norm(r)) for r in ranks], s=55, edgecolor="black", linewidth=0.4)
    for _, row in order.iterrows():
        axes[1].annotate(f"{int(row.init_idx):03d}", (row["rank"], row["final_loss"]),
                         fontsize=6, textcoords="offset points", xytext=(4, 3))
    axes[1].set(title="Final total loss by rank", xlabel="fit rank (1 = best)", ylabel="final loss")
    axes[1].set_yscale("log")
    fig.tight_layout()
    display_figure(fig)


## 7 · Cross-initialization consistency of currents and geometry

Because hidden units are permutable/sign-ambiguous across initializations, raw weights
are not directly comparable. We therefore compare **functional, rotation-invariant
summaries** of the fitted dynamics:

1. **Inter-regional current magnitude** — the time course of `‖W_rec[target,source] · h_source(t)‖`
   for every source→target pair and condition. Comparable across inits.
2. **Relative source contribution** — each source region's signed contribution to the
   within-fixation drive direction of each target (the `relative_projection` used in the
   interregional-current-geometry notebook), for within-region and inter-regional sources.
3. **Latent geometry (RSA)** — per region, the representational dissimilarity matrix (RDM)
   of the recurrent drive across condition×time. The RDM is invariant to rotation/permutation
   of hidden units, so second-order RSA correlations measure geometric consistency.

For each family we build an initialization×initialization correlation matrix; the mean
off-diagonal correlation is the **consistency score** (1 = identical dynamics across inits).

In [14]:
def region_slice(replay, region):
    start, stop = replay["model"].mrnn.get_region_indices(region)
    return slice(int(start), int(stop))


def current_norm_vector(replay):
    """Flattened ‖current‖ over (source, target, condition, time). Rotation/permutation invariant."""
    current_vectors = extract_region_current_vectors(replay)
    parts = []
    for src in region_order:
        for tgt in region_order:
            cur = current_vectors[(src, tgt)].numpy()          # (cond, time, tgt_units)
            parts.append(np.linalg.norm(cur, axis=-1).ravel())  # (cond*time,)
    return np.concatenate(parts)


def relative_contribution_table(replay):
    latent = extract_fixation_latent_dynamics(replay)
    current_vectors = extract_region_current_vectors(replay)
    rows = []
    for active in condition_order:
        a = condition_order.index(active)
        for tgt in region_order:
            sl = region_slice(replay, tgt)
            hidden = latent[active]["hidden_state"][:, sl].numpy()
            drive = latent[active]["recurrent_drive"][:, sl].numpy()
            direction = drive - hidden
            norm = np.linalg.norm(direction, axis=-1, keepdims=True)
            unit = np.divide(direction, np.maximum(norm, 1e-8),
                             out=np.zeros_like(direction), where=norm > 1e-8)
            proj = {src: np.sum(current_vectors[(src, tgt)].numpy()[a] * unit, axis=-1)
                    for src in region_order}
            denom = np.sum([np.abs(v) for v in proj.values()], axis=0)
            denom = np.where(denom > 1e-8, denom, 1.0)
            for src in region_order:
                rel = proj[src] / denom
                for t, value in enumerate(proj[src]):
                    rows.append({"active_condition": active, "source_region": src,
                                 "target_region": tgt, "time_idx": int(t),
                                 "time_s": float(timeline[t]),
                                 "projection": float(value),
                                 "relative_projection": float(rel[t])})
    return pd.DataFrame(rows)


def relative_contribution_vector(rel_table):
    ordered = rel_table.sort_values(["active_condition", "target_region", "source_region", "time_idx"])
    return ordered["relative_projection"].to_numpy(dtype=float)


def drive_rdm_upper(replay, region):
    latent = extract_fixation_latent_dynamics(replay)
    sl = region_slice(replay, region)
    drive = np.stack([latent[c]["recurrent_drive"][:, sl].numpy() for c in condition_order], axis=0)
    X = drive.reshape(-1, drive.shape[-1])           # (cond*time, units)
    Xc = X - X.mean(axis=1, keepdims=True)
    denom = np.linalg.norm(Xc, axis=1, keepdims=True)
    Xn = np.divide(Xc, np.maximum(denom, 1e-8), out=np.zeros_like(Xc), where=denom > 1e-8)
    corr = np.clip(Xn @ Xn.T, -1.0, 1.0)
    rdm = 1.0 - corr
    iu = np.triu_indices(rdm.shape[0], k=1)
    return rdm[iu]


def mean_offdiag(corr):
    n = corr.shape[0]
    if n < 2:
        return np.nan
    iu = np.triu_indices(n, k=1)
    return float(np.nanmean(corr[iu]))


In [15]:
if ANALYSIS_READY:
    ordered_inits = model_rank.sort_values("rank")["init_idx"].astype(int).tolist()

    # Precompute per-init feature vectors and relative-contribution tables.
    current_vecs = {}
    rel_tables = {}
    rel_vecs = {}
    rdm_vecs = {r: {} for r in region_order}
    for init_idx in ordered_inits:
        replay = load_replay(init_idx)
        current_vecs[init_idx] = current_norm_vector(replay)
        rel_tables[init_idx] = relative_contribution_table(replay)
        rel_vecs[init_idx] = relative_contribution_vector(rel_tables[init_idx])
        for region in region_order:
            rdm_vecs[region][init_idx] = drive_rdm_upper(replay, region)

    def corr_matrix(vec_dict):
        M = np.vstack([vec_dict[i] for i in ordered_inits])
        return np.corrcoef(M)

    current_corr = corr_matrix(current_vecs)
    rel_corr = corr_matrix(rel_vecs)
    geometry_corr_by_region = {r: corr_matrix(rdm_vecs[r]) for r in region_order}
    geometry_corr = np.nanmean(np.stack(list(geometry_corr_by_region.values())), axis=0)

    consistency_summary = pd.DataFrame([
        {"metric_family": "inter-regional current magnitude", "consistency_score": mean_offdiag(current_corr)},
        {"metric_family": "relative source contribution", "consistency_score": mean_offdiag(rel_corr)},
        {"metric_family": "latent geometry (drive RDM / RSA)", "consistency_score": mean_offdiag(geometry_corr)},
        *[{"metric_family": f"geometry RSA — {r}", "consistency_score": mean_offdiag(geometry_corr_by_region[r])}
          for r in region_order],
    ])
    display(consistency_summary.round(4))


In [16]:
if ANALYSIS_READY:
    labels = [f"{i:03d}" for i in ordered_inits]
    panels = [
        ("Inter-regional current magnitude", current_corr),
        ("Relative source contribution", rel_corr),
        ("Latent geometry (drive RSA)", geometry_corr),
    ]
    fig, axes = plt.subplots(1, 3, figsize=(15.0, 4.6))
    for ax, (title, corr) in zip(axes, panels):
        im = ax.imshow(corr, vmin=0.0, vmax=1.0, cmap="magma")
        ax.set_xticks(range(len(labels)))
        ax.set_xticklabels(labels, rotation=90, fontsize=6)
        ax.set_yticks(range(len(labels)))
        ax.set_yticklabels(labels, fontsize=6)
        ax.set_title(f"{title}\nmean off-diag = {mean_offdiag(corr):.3f}", fontsize=9)
        fig.colorbar(im, ax=ax, fraction=0.046)
    fig.suptitle("Initialization × initialization consistency (ordered best→worst fit)", y=1.03)
    fig.tight_layout()
    display_figure(fig)


### 7a · Inter-regional current magnitude across inits

Mean current magnitude time course per source→target pair, one line per initialization
(coloured best→worst). Tight overlap ⇒ the rank-1 bottleneck routes a consistent amount
of inter-regional drive regardless of initialization.

In [17]:
if ANALYSIS_READY:
    cmap = plt.get_cmap("viridis")
    norm = Normalize(vmin=1, vmax=len(ordered_inits))
    rank_of = dict(zip(model_rank["init_idx"].astype(int), model_rank["rank"].astype(int)))
    off_pairs = [(s, t) for s in region_order for t in region_order if s != t]
    ncol = len(region_order) - 1
    fig, axes = plt.subplots(len(region_order), ncol, figsize=(3.0 * ncol, 2.2 * len(region_order)),
                             squeeze=False, sharex=True)
    for r_i, tgt in enumerate(region_order):
        srcs = [s for s in region_order if s != tgt]
        for c_i, src in enumerate(srcs):
            ax = axes[r_i, c_i]
            for init_idx in ordered_inits:
                cur = extract_region_current_vectors(load_replay(init_idx))[(src, tgt)].numpy()
                mag = np.linalg.norm(cur, axis=-1).mean(axis=0)   # mean over conditions -> (time,)
                ax.plot(timeline, mag, color=cmap(norm(rank_of[init_idx])), lw=1.0, alpha=0.75)
            ax.axvline(0.0, color="0.5", lw=0.6)
            ax.set_title(f"{src}→{tgt}", fontsize=8)
            if r_i == len(region_order) - 1:
                ax.set_xlabel("time (s)")
            if c_i == 0:
                ax.set_ylabel("‖current‖")
    sm = ScalarMappable(cmap=cmap, norm=norm); sm.set_array([])
    fig.colorbar(sm, ax=axes, fraction=0.02, label="fit rank (1 = best)")
    fig.suptitle("Inter-regional current magnitude — every initialization", y=1.02)
    display_figure(fig)


### 7b · Relative source contribution — mean ± spread across inits

For each active fixation condition and target region, the mean (line) and ±1 SD (band)
across initializations of each source region's relative contribution to the target's
within-fixation drive. Narrow bands ⇒ consistent circuit logic across inits.

In [18]:
if ANALYSIS_READY:
    stacked = pd.concat(
        [rel_tables[i].assign(init_idx=i) for i in ordered_inits], ignore_index=True
    )
    agg = (stacked.groupby(["active_condition", "target_region", "source_region", "time_idx", "time_s"],
                           as_index=False)
           .agg(mean_rel=("relative_projection", "mean"), sd_rel=("relative_projection", "std")))
    agg["sd_rel"] = agg["sd_rel"].fillna(0.0)

    fig, axes = plt.subplots(len(condition_order), len(region_order),
                             figsize=(13.0, 8.2), squeeze=False, sharex=True, sharey=True)
    for r_i, active in enumerate(condition_order):
        for c_i, tgt in enumerate(region_order):
            ax = axes[r_i, c_i]
            sub = agg[(agg["active_condition"] == active) & (agg["target_region"] == tgt)]
            ax.axhline(0.0, color="black", lw=0.6, alpha=0.5)
            ax.axvline(0.0, color="0.5", lw=0.6, alpha=0.45)
            for src in region_order:
                tr = sub[sub["source_region"] == src].sort_values("time_idx")
                x = tr["time_s"].to_numpy(); y = tr["mean_rel"].to_numpy(); sd = tr["sd_rel"].to_numpy()
                color = region_colors.get(src, "0.3")
                ax.plot(x, y, color=color, lw=1.2, label=src)
                ax.fill_between(x, y - sd, y + sd, color=color, alpha=0.18)
            ax.set_ylim(-1.0, 1.0)
            if r_i == 0:
                ax.set_title(f"target {tgt}", fontsize=9)
            if c_i == 0:
                ax.set_ylabel(f"{active}\nrel. contribution")
            if r_i == len(condition_order) - 1:
                ax.set_xlabel("time (s)")
    axes[0, -1].legend(frameon=False, fontsize=7, loc="upper left", bbox_to_anchor=(1.02, 1.0))
    fig.suptitle("Relative source contribution — mean ± SD across initializations", y=1.02)
    fig.tight_layout()
    display_figure(fig)


### 7c · Does fit quality predict dynamical consistency?

Relate each initialization's fit-quality rank to how similar its dynamics are to the
**best** model (row/column of the correlation matrices above). If well-fit models are
also the most mutually consistent, the good fits converge on a shared solution.

In [19]:
if ANALYSIS_READY:
    best_pos = ordered_inits.index(best_init_idx)
    sim_to_best = pd.DataFrame({
        "init_idx": ordered_inits,
        "rank": [rank_of[i] for i in ordered_inits],
        "current_sim_to_best": current_corr[best_pos],
        "relcontrib_sim_to_best": rel_corr[best_pos],
        "geometry_sim_to_best": geometry_corr[best_pos],
    })
    display(sim_to_best.round(4))

    fig, ax = plt.subplots(figsize=(7.6, 4.0))
    for column, marker, label in [
        ("current_sim_to_best", "o", "current magnitude"),
        ("relcontrib_sim_to_best", "s", "relative contribution"),
        ("geometry_sim_to_best", "^", "latent geometry"),
    ]:
        sub = sim_to_best[sim_to_best["init_idx"] != best_init_idx]
        ax.scatter(sub["rank"], sub[column], marker=marker, s=48, alpha=0.8, label=label)
    ax.set(title="Similarity to best-fit model vs fit rank",
           xlabel="fit rank (1 = best)", ylabel="correlation with best model")
    ax.legend(frameon=False, fontsize=8)
    fig.tight_layout()
    display_figure(fig)


## 8 · Persist ensemble summaries

In [20]:
if ANALYSIS_READY:
    summary_dir = ensemble_dir / "ensemble_summaries"
    summary_dir.mkdir(parents=True, exist_ok=True)
    run_table.to_csv(summary_dir / "run_table.csv", index=False)
    model_rank.to_csv(summary_dir / "model_rank.csv", index=False)
    consistency_summary.to_csv(summary_dir / "consistency_summary.csv", index=False)
    sim_to_best.to_csv(summary_dir / "similarity_to_best.csv", index=False)
    np.savez(
        summary_dir / "consistency_correlations.npz",
        ordered_inits=np.asarray(ordered_inits),
        current_corr=current_corr,
        rel_corr=rel_corr,
        geometry_corr=geometry_corr,
    )
    print("Saved ensemble summaries to", summary_dir)
